# Stage 2: Mistral-only scoring, BOTH T4s (dual-process split)

Per status.md's hardware-split plan: Stage 1 (`scripts/run_qwen_local.py`) runs
entirely on the local 4060 -- routing, beam-k10 Qwen candidate generation, Qwen's own
prefix-cache teacher-forced scoring, KN5 scoring -- and writes `dev_qwen_scores.jsonl`
/ `test_qwen_scores.jsonl`.

This notebook does ONLY the Mistral side, split across BOTH of the session's T4s
(Kaggle bills the whole T4x2 session regardless of how many GPUs the code actually
touches, so leaving the second one idle is free compute left on the table). Two
SEPARATE OS PROCESSES (not threads -- separate CUDA contexts, sidesteps the exact
thread-safety issues that crashed the earlier ThreadPoolExecutor attempt, see
status.md's v6-v9 crash log), each loading its own Mistral instance pinned to one
GPU, each scoring half the word rows (`index % 2 == shard`) via the same
prefix-KV-cache method verified in `scripts/score_candidates_prefix_cache.py`.
Writes `dev_mistral_scores.jsonl` / `test_mistral_scores.jsonl` in the original row
order -- same output format as the single-GPU version, `combine_final.py` unchanged.

In [ ]:
import os, glob, json, subprocess, sys, time

import torch
print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i}", torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")
NUM_SHARDS = torch.cuda.device_count() if torch.cuda.device_count() >= 2 else 1

In [ ]:
def find_file(name):
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert matches, f"{name} not found under /kaggle/input -- check the dataset is attached"
    return matches[0]

def find_by_config(model_type):
    for cfg_path in glob.glob("/kaggle/input/**/config.json", recursive=True):
        try:
            cfg = json.load(open(cfg_path))
        except (json.JSONDecodeError, OSError):
            continue
        if cfg.get("model_type") == model_type:
            return os.path.dirname(cfg_path)
    return None

DEV_SCORES_PATH = find_file("dev_qwen_scores.jsonl")
TEST_SCORES_PATH = find_file("test_qwen_scores.jsonl")
MISTRAL_BASE = find_by_config("mistral") or "mistralai/Mistral-7B-v0.1"
print("DEV_SCORES_PATH:", DEV_SCORES_PATH)
print("TEST_SCORES_PATH:", TEST_SCORES_PATH)
print("MISTRAL_BASE:", MISTRAL_BASE)
print("NUM_SHARDS:", NUM_SHARDS)

## Materialize the worker script -- self-contained, one process/GPU/Mistral-instance

Written to `/kaggle/working/` at runtime rather than attached as a separate file --
keeps this a single-notebook artifact, matching every other notebook this session
(inlined helpers, no cross-file imports Kaggle can't resolve). Inlines
`score_candidates_prefix_cache` verbatim from `scripts/score_candidates_prefix_cache.py`
(verified against a real model there, diff ~1e-6) and `_expand_cache`.

In [ ]:
WORKER_SRC = r'''
import argparse, json, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def _expand_cache(past_key_values, n):
    if hasattr(past_key_values, "batch_repeat_interleave"):
        past_key_values.batch_repeat_interleave(n)
        return past_key_values
    if hasattr(past_key_values, "key_cache"):
        for i in range(len(past_key_values.key_cache)):
            past_key_values.key_cache[i] = past_key_values.key_cache[i].repeat_interleave(n, dim=0)
            past_key_values.value_cache[i] = past_key_values.value_cache[i].repeat_interleave(n, dim=0)
        return past_key_values
    return tuple((k.repeat_interleave(n, dim=0), v.repeat_interleave(n, dim=0)) for k, v in past_key_values)

@torch.inference_mode()
def score_candidates_prefix_cache(model, tokenizer, context, candidates, device):
    if not candidates:
        return []
    ctx_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    cand_id_lists = [tokenizer(" " + cand, add_special_tokens=False)["input_ids"] for cand in candidates]
    prefix_len = len(ctx_ids)
    n = len(candidates)
    pad_id = tokenizer.pad_token_id

    ctx_input = torch.tensor([ctx_ids], device=device)
    out1 = model(input_ids=ctx_input, use_cache=True)
    past = _expand_cache(out1.past_key_values, n)
    first_tok_logprob_row = torch.log_softmax(out1.logits[0, -1].float(), dim=-1)

    cand_lens = [len(ids) for ids in cand_id_lists]
    max_len = max(cand_lens)
    cont_ids = torch.full((n, max_len), pad_id, dtype=torch.long)
    cont_mask = torch.zeros((n, max_len), dtype=torch.long)
    for i, ids in enumerate(cand_id_lists):
        cont_ids[i, :len(ids)] = torch.tensor(ids)
        cont_mask[i, :len(ids)] = 1
    cont_ids, cont_mask = cont_ids.to(device), cont_mask.to(device)

    full_attn_mask = torch.cat([torch.ones(n, prefix_len, dtype=torch.long, device=device), cont_mask], dim=1)
    position_ids = torch.arange(prefix_len, prefix_len + max_len, device=device).unsqueeze(0).expand(n, -1)

    if max_len > 1:
        out2 = model(input_ids=cont_ids, attention_mask=full_attn_mask, past_key_values=past,
                      position_ids=position_ids, use_cache=False)
        cont_logprobs = torch.log_softmax(out2.logits[:, :-1], dim=-1)
    else:
        cont_logprobs = None

    results = []
    for i, ids in enumerate(cand_id_lists):
        lp = first_tok_logprob_row[ids[0]].float().item()
        for t in range(1, len(ids)):
            lp += cont_logprobs[i, t - 1, ids[t]].float().item()
        results.append(lp)
    return results

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def score_shard(rows, model, tok, device, shard, num_shards, name):
    t0 = time.time()
    out = []
    assigned = [i for i in range(len(rows)) if i % num_shards == shard
                and rows[i]["route"] == "word" and rows[i].get("candidates")]
    for j, i in enumerate(assigned):
        r = rows[i]
        scores = score_candidates_prefix_cache(model, tok, r["context"], r["candidates"], device)
        out.append({"index": i, "mistral_scores": scores})
        if (j + 1) % 200 == 0 or j + 1 == len(assigned):
            el = time.time() - t0
            print(f"[{name} shard{shard} {j+1}/{len(assigned)}] {el:.1f}s, {(j+1)/max(el,1e-9):.2f} rows/s", flush=True)
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dev-jsonl", required=True)
    ap.add_argument("--test-jsonl", required=True)
    ap.add_argument("--mistral-base", required=True)
    ap.add_argument("--device", required=True)
    ap.add_argument("--shard", type=int, required=True)
    ap.add_argument("--num-shards", type=int, required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()

    tok = AutoTokenizer.from_pretrained(args.mistral_base)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        args.mistral_base, dtype=torch.float16, attn_implementation="sdpa").to(args.device)
    model.eval()
    print(f"[shard{args.shard}] Mistral loaded fp16/sdpa on {args.device} in {time.time()-t0:.1f}s", flush=True)

    dev_rows = load_jsonl(args.dev_jsonl)
    test_rows = load_jsonl(args.test_jsonl)
    if args.limit:
        dev_rows, test_rows = dev_rows[:args.limit], test_rows[:args.limit]

    dev_out = score_shard(dev_rows, model, tok, args.device, args.shard, args.num_shards, "dev")
    test_out = score_shard(test_rows, model, tok, args.device, args.shard, args.num_shards, "test")

    with open(f"{args.out_dir}/dev_mistral_shard{args.shard}.jsonl", "w", encoding="utf-8") as f:
        for r in dev_out:
            f.write(json.dumps(r) + "\n")
    with open(f"{args.out_dir}/test_mistral_shard{args.shard}.jsonl", "w", encoding="utf-8") as f:
        for r in test_out:
            f.write(json.dumps(r) + "\n")
    print(f"[shard{args.shard}] wrote {len(dev_out)} dev + {len(test_out)} test scored rows", flush=True)

if __name__ == "__main__":
    main()
'''
with open("/kaggle/working/mistral_worker.py", "w", encoding="utf-8") as f:
    f.write(WORKER_SRC)
print("wrote /kaggle/working/mistral_worker.py")

In [ ]:
SMALL_BATCH_LIMIT = None  # real run

def launch_shard(shard, device):
    cmd = [sys.executable, "/kaggle/working/mistral_worker.py",
           "--dev-jsonl", DEV_SCORES_PATH, "--test-jsonl", TEST_SCORES_PATH,
           "--mistral-base", MISTRAL_BASE, "--device", device,
           "--shard", str(shard), "--num-shards", str(NUM_SHARDS),
           "--out-dir", "/kaggle/working"]
    if SMALL_BATCH_LIMIT:
        cmd += ["--limit", str(SMALL_BATCH_LIMIT)]
    return subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

t0 = time.time()
procs = [launch_shard(s, f"cuda:{s}") for s in range(NUM_SHARDS)]
print(f"launched {len(procs)} shard process(es), one per GPU", flush=True)

# stream both processes' output interleaved as it arrives, don't block on one before
# starting the other -- that's the whole point, they must run CONCURRENTLY
import selectors
sel = selectors.DefaultSelector()
for p in procs:
    sel.register(p.stdout, selectors.EVENT_READ, p)
remaining = set(procs)
while remaining:
    for key, _ in sel.select(timeout=1):
        line = key.fileobj.readline()
        if line:
            print(line.rstrip(), flush=True)
    for p in list(remaining):
        if p.poll() is not None:
            remaining.discard(p)

returncodes = [p.wait() for p in procs]
print(f"\nall shards done in {time.time()-t0:.1f}s, returncodes={returncodes}")
assert all(rc == 0 for rc in returncodes), f"a shard process failed: {returncodes}"

## Merge shards back into full-length dev/test files -- same format combine_final.py expects

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def merge_shards(name, n_rows):
    by_index = {}
    for s in range(NUM_SHARDS):
        for r in load_jsonl(f"/kaggle/working/{name}_mistral_shard{s}.jsonl"):
            by_index[r["index"]] = r["mistral_scores"]
    return [{"mistral_scores": by_index.get(i)} for i in range(n_rows)]

dev_in = load_jsonl(DEV_SCORES_PATH)
test_in = load_jsonl(TEST_SCORES_PATH)
if SMALL_BATCH_LIMIT:
    dev_in, test_in = dev_in[:SMALL_BATCH_LIMIT], test_in[:SMALL_BATCH_LIMIT]

dev_out = merge_shards("dev", len(dev_in))
test_out = merge_shards("test", len(test_in))

with open("/kaggle/working/dev_mistral_scores.jsonl", "w", encoding="utf-8") as f:
    for r in dev_out:
        f.write(json.dumps(r) + "\n")
with open("/kaggle/working/test_mistral_scores.jsonl", "w", encoding="utf-8") as f:
    for r in test_out:
        f.write(json.dumps(r) + "\n")
print(f"wrote dev_mistral_scores.jsonl ({len(dev_out)}), test_mistral_scores.jsonl ({len(test_out)}) "
      f"(SMALL_BATCH_LIMIT={SMALL_BATCH_LIMIT} -- NOT the full run until that's None)")